In [179]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
import scipy as sp
import scipy.stats as stats
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy
import squarify
try:
    import piplite
    await piplite.install(['ipywidgets'])
except ImportError:
    pass
import ipywidgets as widgets
import json

today = date.today()

**building 1kb regions**

In [3]:
chrm_lengths = [248956422, 242193529, 198295559, 190214555, 181538259, 170805979, 159345973, 145138636, 138394717, 133797422, 135086622, 133275309, 114364328, 107043718, 101991189, 90338345, 83257441, 80373285, 58617616, 64444167, 46709983, 50818468, 156040895, 57227415]
chrms = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', 'X', 'Y']

In [47]:
bed_out = []
for c, l in zip(chrms, chrm_lengths):
    start = 0
    end = 1000
    seqname = 'chr' + c
    while start < l:
        bed_out.append([seqname, start, end])
        start += 1000
        end += 1000
bed_out = np.array(bed_out)

In [48]:
bed_out.shape

(3088281, 3)

In [49]:
out_f = '/home/ctrader/region_similarity/data/rank_based/hg38.1kb.regions.CT20230323.bed'
np.savetxt(out_f, bed_out, fmt='%s', delimiter='\t')

**combining datasets and reducing precision**

In [88]:
def load_per(per_f):
    hdf5_f = h5py.File(per_f, 'r')
    percentile = hdf5_f['percentile'][:]
    factors = np.char.decode(hdf5_f['factors'][:])
    color = np.char.decode(hdf5_f['colors'][:])
    hdf5_f.close()
    return percentile, factors, color

In [89]:
datadir = '/home/ctrader/region_similarity/data/rank_based/'
DHS_per_f = datadir + 'regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank_percentile.hdf5'
epi_per_f = datadir + 'regions_1kb_CT20230323.epilogos.statesum_rank_percentile.hdf5'

# load region percentiles
DHS_per, DHS_factors, DHS_color = load_per(DHS_per_f)
epi_per, epi_factors, epi_color = load_per(epi_per_f)

In [90]:
percents = np.hstack((DHS_per, epi_per))
factor_order = np.append(DHS_factors, epi_factors)
metric_order = np.append(np.array(['DHSs'] * len(DHS_factors)),
                         np.array(['chromatin states'] *
                                  len(epi_factors)))
colors = np.append(DHS_color, epi_color)

In [91]:
percents.shape

(3088281, 34)

In [92]:
# datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
# out_f = datadir + 'regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'
# print(out_f)
# np.save(out_f, percents.astype('float32'))

/home/ctrader/AG/NarrativeGenomics/region_narration/data/regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy


In [100]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
per_f = datadir + 'regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'

t = time.time()
percents = np.load(per_f)
time.time() - t

1.3436346054077148

In [96]:
datadir = '/home/ctrader/region_similarity/data/rank_based/'
regions_path = datadir + 'hg38.1kb.regions.CT20230323.sorted.bed'

f = open(regions_path, 'r')
regions_sep = []
for line in f.readlines():
    line = line.strip().split('\t')
    regions_sep.append(line)
f.close()
regions_sep = np.array(regions_sep)

In [97]:
regions_sep.shape

(3088281, 3)

In [98]:
# datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
# out_f = datadir + 'hg38.1kb.regions.CT20230323.sorted.bed.npy'

# print(out_f)
# np.save(out_f, regions_sep)

/home/ctrader/AG/NarrativeGenomics/region_narration/data/hg38.1kb.regions.CT20230323.sorted.bed.npy


In [99]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
regions_path = datadir + 'hg38.1kb.regions.CT20230323.sorted.bed.npy'

t = time.time()
regions_sep = np.load(regions_path)
time.time() - t

0.8398733139038086

**Rank aggregation**

Load data

In [105]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
per_f = datadir + 'regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'

t = time.time()
percents1 = np.load(per_f)
print(time.time() - t)

print(percents1.shape)

0.2738511562347412
(3088281, 34)


In [106]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
per_f = datadir + 'regions_25kb_CT20230321.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'

t = time.time()
percents25 = np.load(per_f)
print(time.time() - t)

print(percents25.shape)

0.03305935859680176
(88036, 34)


In [107]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
regions_path = datadir + 'hg38.1kb.regions.CT20230323.sorted.bed.npy'

t = time.time()
regions_sep1 = np.load(regions_path)
print(time.time() - t)

print(regions_sep1.shape)

0.23842215538024902
(3088281, 3)


In [108]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
regions_path = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.bed.npy'

t = time.time()
regions_sep25 = np.load(regions_path)
print(time.time() - t)

print(regions_sep25.shape)

0.029695510864257812
(88036, 3)


find testing region

In [141]:
seqname25 = regions_sep25[:, 0]
start25 = seqname = regions_sep25[:, 1].astype(int)
end25 = seqname = regions_sep25[:, 2].astype(int)

test_regions_25 = np.where((start25 % 1000 == 0) & (end25 % 1000 == 0) & (end25 - start25 == 25000))[0]
test_regions_25

array([ 1034,  1035,  7365,  7366,  7367,  7373,  7378,  7389,  7395,
       11024, 11030, 11034, 11048, 11052, 13900, 13903, 13914, 13917,
       13921, 13922, 27450, 27451, 27457, 27458, 27527, 27528, 27529,
       32226, 32227, 32228, 32231, 32232, 32233, 32240, 32311, 32312,
       32313, 32314, 32315, 32316, 32321, 32324, 32325, 32330, 32331,
       32333, 32379, 32425, 32436, 32445, 32446, 34557, 34564, 34566,
       34567, 34568, 34572, 34573, 34574, 34575, 34576, 34582, 34585,
       34586, 34587, 34595, 34596, 34598, 34603, 34604, 34606, 34607,
       34617, 34625, 34626, 34629, 34630, 34631, 34633, 34634, 34635,
       34639, 34640, 34644, 34651, 36356, 36369, 65181, 65182, 65189,
       65190, 65191, 65197, 65204, 65205, 65215, 65216, 65217, 65233,
       65234, 70430, 71606, 71611, 71627, 71628, 74679, 74680, 74681,
       74682, 74683, 74688, 74689, 74697, 74711, 74712, 74713, 74720,
       74721, 75266, 75297, 79710, 79719, 79731, 79732, 79746, 79747,
       79761, 79767,

In [140]:
r25 = 11030

print(r25)
print(regions_sep25[r25])
print(percents25[r25])

11030
['chr11' '266000' '291000']
[0.77587026 0.9909431  0.9638975  0.4045784  0.66870534 0.525814
 0.87903553 0.40737462 0.95267886 0.39017025 0.43827942 0.47864032
 0.4180047  0.73398066 0.91926354 0.3728135  0.60011256 0.9568963
 0.864435   0.8903498  0.         0.         0.46315452 0.7292932
 0.6549117  0.8799299  0.7187846  0.38982424 0.06263918 0.9383826
 0.9583396  0.9486281  0.8254282  0.        ]


In [ ]:
chr11:266000-291000

In [155]:
seqname1 = regions_sep1[:, 0]
start1 = seqname = regions_sep1[:, 1].astype(int)
end1 = seqname = regions_sep1[:, 2].astype(int)

r1_coor = np.where((seqname1 == regions_sep25[r25][0]) & 
         (start1 >= int(regions_sep25[r25][1])) & 
         (end1 <= int(regions_sep25[r25][2])))[0]
r1_coor

array([383021, 383022, 383023, 383024, 383025, 383026, 383027, 383028,
       383029, 383030, 383031, 383032, 383033, 383034, 383035, 383036,
       383037, 383038, 383039, 383040, 383041, 383042, 383043, 383044,
       383045])

25kb ranks

In [166]:
print(factor_order)

['Placental_trophoblast' 'Lymphoid' 'Myeloid_erythroid' 'Cardiac'
 'Musculoskeletal' 'Vascular_endothelial' 'Primitive_embryonic' 'Neural'
 'Digestive' 'StromalA' 'StromalB' 'Renal_cancer' 'Cancer_epithelial'
 'Pulmonarydevel' 'Organdevel_renal' 'Tissueinvariant' 'TssA' 'TssFlnk'
 'TssFlnkU' 'TssFlnkD' 'Tx' 'TxWk' 'EnhG1' 'EnhG2' 'EnhA1' 'EnhA2' 'EnhWk'
 'ZNF_Rpts' 'Het' 'TssBiv' 'EnhBiv' 'ReprPC' 'ReprPCWk' 'Quies']


In [176]:
# 25kb percents
original = percents25[r25]
print(original)

[0.77587026 0.9909431  0.9638975  0.4045784  0.66870534 0.525814
 0.87903553 0.40737462 0.95267886 0.39017025 0.43827942 0.47864032
 0.4180047  0.73398066 0.91926354 0.3728135  0.60011256 0.9568963
 0.864435   0.8903498  0.         0.         0.46315452 0.7292932
 0.6549117  0.8799299  0.7187846  0.38982424 0.06263918 0.9383826
 0.9583396  0.9486281  0.8254282  0.        ]


In [195]:
np.sum(np.abs(original - mean1))

11.275668

max

In [187]:
# 1kb max percents
max1 = np.max(percents1[r1_coor], axis=0)
print(max1)
print()
print(euclidean_distances(np.vstack((original, max1)))[0, 1])

[0.84464574 0.93460834 0.9152938  0.77515733 0.72526664 0.77247757
 0.8794308  0.6259689  0.9397785  0.7948197  0.67379767 0.7178014
 0.7555736  0.9523956  0.93768007 0.639939   0.7372863  0.9867282
 0.8746137  0.9687057  0.         0.35160813 0.6829601  0.7804768
 0.83536303 0.9661574  0.96565026 0.6551687  0.5519648  0.9370716
 0.9572669  0.95388633 0.96889246 0.        ]

1.1837486


mean

In [186]:
# 1kb mean percents
mean1 = np.mean(percents1[r1_coor], axis=0)
print(mean1)
print()
print(euclidean_distances(np.vstack((original, mean1)))[0, 1])

[0.5623915  0.6619558  0.50375015 0.1160651  0.34822848 0.15469883
 0.7253644  0.15296373 0.58279794 0.15148182 0.12696414 0.2397685
 0.19102956 0.20895727 0.5520675  0.12032104 0.1844312  0.31882897
 0.30178347 0.40473625 0.         0.01805979 0.09975045 0.18880719
 0.19729412 0.38444576 0.22990152 0.06054427 0.02298138 0.32067367
 0.6074202  0.6656314  0.55751634 0.        ]

2.1606467


In [196]:
percents1[r1_coor][:, 0]

array([0.6445767 , 0.84464574, 0.58357245, 0.73459154, 0.68594337,
       0.6908079 , 0.48729613, 0.33741263, 0.5583597 , 0.5989402 ,
       0.33343443, 0.2754232 , 0.40092263, 0.4667766 , 0.635789  ,
       0.40598762, 0.5859644 , 0.71265304, 0.72176725, 0.68757343,
       0.57733613, 0.44696853, 0.58937645, 0.68629205, 0.36737713],
      dtype=float32)

In [203]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
gz_file_path = datadir + 'hg38.1kb.regions.CT20230323.sorted.rownum4.bed.gz'
genomic_location = 'chr1:198636001-198636002'

In [204]:
# load roi
gz_tb = tabix.open(gz_file_path)
roi = np.array(
    list(gz_tb.querys(genomic_location))
)[:, 3].astype(int)[0]

In [205]:
list(gz_tb.querys(genomic_location))

[['chr1', '198636000', '198637000 ', '198636']]

In [206]:
percents1[198636]

array([0.        , 0.8930147 , 0.09256393, 0.30258968, 0.        ,
       0.        , 0.36433786, 0.4688635 , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.44849995, 0.        , 0.        , 0.51083887, 0.2779842 ,
       0.        , 0.        , 0.        , 0.        , 0.824125  ,
       0.6912791 , 0.41134906, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.06012321], dtype=float32)